# Closed-model clinical answers

**Goal:** Inspect API payloads and load the completed clinical results.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Inspect the provider settings

The clinical comparison used Gemini 3.8 Flash, GPT 5.5 and Opus 4.8. Provider interfaces have different decoding controls; no claim of identical internal precision or reasoning is made. Credentials belong in environment variables only.

In [ ]:
from sleepinn_study.providers import generation_payload, request_json, answer_text
case = read_jsonl(ROOT / "data/final/clinical.jsonl")[0]
from sleepinn_study.workflow import answer_prompt
payload = generation_payload("openai", "gpt-5.5-2026-04-23", answer_prompt(case))
display(payload)
display(read_json(ROOT / "data/config/clinical_execution_protocol.json"))
closed_settings = read_json(ROOT / "data/config/closed_generation_settings.json")
for provider, settings in closed_settings.items():
    display(generation_payload(provider, settings["model"], answer_prompt(case),
                               opus_settings=settings["settings"] if provider == "openrouter" else None))


## 3. Make a request only when intended

The helper saves a pending marker before each request. On a timeout it preserves the incident and does not retry automatically, because the provider may already have processed the call. Request receipts remain under ignored outputs/.

In [ ]:
RUN_API = False
if RUN_API:
    response = request_json("openai", "gpt-5.5-2026-04-23", payload,
                            ROOT / "outputs/closed_new_001/request_000", enabled=True)
    print(answer_text("openai", response))
else:
    print("API execution disabled.")

## 4. Check the saved closed-model population

Each closed model answered 120 cases in both conditions.

In [ ]:
from sleepinn_study.io import answer_frame
answers = answer_frame("clinical")
display(answers[answers.precision.eq("API")].groupby(["model_id", "mode"]).size().rename("answers"))